# Play Clean Up

Run the final cell to open the game window and round controls. Choose one action for every agent, from `player_0` through `player_n`, then select **Execute simultaneous step**. The environment receives the complete action dictionary in one `step` call.

In [1]:
import ipywidgets as widgets
from IPython.display import display

from masa.envs.multiagent.tabular import CleanUp

SEED = 0
WINDOW_SIZE = 960
ACTION_OPTIONS = [
    ("Choose an action...", None),
    ("0 - No-op", 0),
    ("1 - Move forward", 1),
    ("2 - Strafe right", 2),
    ("3 - Strafe left", 3),
    ("4 - Move backward", 4),
    ("5 - Turn left", 5),
    ("6 - Turn right", 6),
    ("7 - Zap beam", 7),
    ("8 - Cleaning beam", 8),
]


In [2]:
env = CleanUp(render_mode="human", render_window_size=WINDOW_SIZE)
observations, infos = env.reset(seed=SEED)

action_inputs = {
    agent: widgets.Dropdown(
        options=ACTION_OPTIONS,
        value=None,
        description=agent,
        layout=widgets.Layout(width="430px"),
        style={"description_width": "90px"},
    )
    for agent in env.possible_agents
}
step_button = widgets.Button(
    description="Execute simultaneous step",
    button_style="success",
    icon="play",
)
reset_button = widgets.Button(description="Reset", icon="refresh")
close_button = widgets.Button(description="Close", button_style="danger", icon="stop")
output = widgets.Output(layout=widgets.Layout(max_height="260px", overflow="auto"))
round_number = 0

def _set_inputs_enabled(enabled):
    for selector in action_inputs.values():
        selector.disabled = not enabled
    step_button.disabled = not enabled

def _clear_actions():
    for selector in action_inputs.values():
        selector.value = None

def _execute_step(_):
    global observations, infos, round_number
    missing = [agent for agent in env.agents if action_inputs[agent].value is None]
    with output:
        if missing:
            print("Choose an action for every agent. Missing:", ", ".join(missing))
            return
        actions = {agent: action_inputs[agent].value for agent in env.agents}
        observations, rewards, terminations, truncations, infos = env.step(actions)
        round_number += 1
        print(f"Round {round_number}:", actions)
        print("Rewards:", rewards)
        unsafe = [agent for agent, obs in observations.items() if env.cost_fn(env.label_fn(obs))]
        if unsafe:
            print("Unsafe agents:", unsafe)
        finished = any(terminations.values()) or any(truncations.values())
        if finished:
            print("Episode finished. Reset to play again.")
            _set_inputs_enabled(False)
    _clear_actions()

def _reset(_):
    global observations, infos, round_number
    observations, infos = env.reset(seed=SEED)
    round_number = 0
    _clear_actions()
    _set_inputs_enabled(True)
    with output:
        output.clear_output()
        print("Environment reset. Select every agent's action for round 1.")

def _close(_):
    env.close()
    _set_inputs_enabled(False)
    reset_button.disabled = True
    close_button.disabled = True
    with output:
        print("Environment closed. Rerun this cell to play again.")

step_button.on_click(_execute_step)
reset_button.on_click(_reset)
close_button.on_click(_close)

controls = widgets.GridBox(
    children=[action_inputs[agent] for agent in env.possible_agents],
    layout=widgets.Layout(grid_template_columns="repeat(2, 430px)", grid_gap="8px 16px"),
)
display(widgets.VBox([
    widgets.HTML("<b>Round actions:</b> every selector is required before the step executes."),
    controls,
    widgets.HBox([step_button, reset_button, close_button]),
    output,
]))
with output:
    print("Select every agent's action for round 1.")
